# Libs | File Paths | Dataset

In [6]:
from pathlib import Path
import timeit
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, recall_score
from tensorflow.lite.python import schema_py_generated as schema_fb

PROJECT_ROOT = Path(r"C:\tinyml-fdia-windows")

KERAS_DIR =  PROJECT_ROOT  / "Models" / "keras"
TFLITE_DIR = PROJECT_ROOT / "models" / "tflite"
DATA_PATH = PROJECT_ROOT / "data" / "fdia_dataset_processed.npz"

data = np.load(DATA_PATH)

X_test = data["X_test"]
y_test = data["y_test"]

# LSTM input: (samples, timesteps, features)
X_test_lstm = np.transpose(X_test, (0, 2, 1)).astype(np.float32)

# Same timing subset used in the Keras benchmark
np.random.seed(42)
sample_size = int(0.25 * len(X_test))
timing_idx = np.random.choice(len(X_test), sample_size, replace=False)

X_timing = X_test_lstm[timing_idx]

print("X_test:", X_test_lstm.shape)
print("Timing samples:", len(X_timing))

X_test: (9720, 83, 6)
Timing samples: 2430


# Model Size Measurement Based on Total Parameters Count

In [2]:
def get_parameter_size(model_path):
    model_bytes = Path(model_path).read_bytes()
    model = schema_fb.Model.GetRootAsModel(model_bytes, 0)

    seen_buffers = set()

    total_params = 0
    total_bytes = 0

    float32_params = 0
    int8_params = 0

    for sg_idx in range(model.SubgraphsLength()):
        subgraph = model.Subgraphs(sg_idx)

        for tensor_idx in range(subgraph.TensorsLength()):
            tensor = subgraph.Tensors(tensor_idx)

            name = tensor.Name()
            name = name.decode("utf-8") if name else ""

            # Trainable weight/bias tensors
            is_parameter = (
                "MatMul" in name
                or "BiasAdd" in name
                or name.endswith("/Add")
            )

            if not is_parameter:
                continue

            buffer_idx = tensor.Buffer()

            if buffer_idx in seen_buffers:
                continue

            buffer = model.Buffers(buffer_idx)

            if buffer is None or buffer.DataLength() == 0:
                continue

            shape = [tensor.Shape(i) for i in range(tensor.ShapeLength())]
            count = int(np.prod(shape)) if shape else 1

            tensor_type = tensor.Type()

            if tensor_type == schema_fb.TensorType.FLOAT32:
                bytes_per_param = 4
                float32_params += count

            elif tensor_type == schema_fb.TensorType.INT8:
                bytes_per_param = 1
                int8_params += count

            else:
                continue

            total_params += count
            total_bytes += count * bytes_per_param

            seen_buffers.add(buffer_idx)

    return {
        "params": total_params,
        "float32_params": float32_params,
        "int8_params": int8_params,
        "size_kb": total_bytes / 1024
    }

In [3]:
test_path = TFLITE_DIR / "LSTM_Original_quant.tflite"

info = get_parameter_size(test_path)

print(f"Parameters:       {info['params']:,}")
print(f"Float32 params:   {info['float32_params']:,}")
print(f"Int8 params:      {info['int8_params']:,}")
print(f"Model size:       {info['size_kb']:.3f} KB")

Parameters:       159,581
Float32 params:   1,593
Int8 params:      157,988
Model size:       160.508 KB


In [9]:
def predict_tflite(model_path, X):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    predictions = np.empty(len(X), dtype=np.float32)

    for i, sample in enumerate(X):
        interpreter.set_tensor(
            input_details[0]["index"],
            sample[np.newaxis, ...].astype(np.float32)
        )

        interpreter.invoke()

        predictions[i] = interpreter.get_tensor(
            output_details[0]["index"]
        ).squeeze()

    return predictions


def measure_inference_time_tflite(model_path, X_timing, runs=100):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_index = interpreter.get_input_details()[0]["index"]

    run_times = []

    for _ in range(runs):
        start = timeit.default_timer()

        for sample in X_timing:
            interpreter.set_tensor(
                input_index,
                sample[np.newaxis, ...].astype(np.float32)
            )
            interpreter.invoke()

        end = timeit.default_timer()

        total_time_ms = (end - start) * 1000
        mean_time_ms = total_time_ms / len(X_timing)

        run_times.append(mean_time_ms)

    return np.mean(run_times)


def evaluate_tflite(model_path):
    predictions = predict_tflite(model_path, X_test_lstm)
    y_pred = (predictions >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)

    fdia_recall = recall_score(
        y_test,
        y_pred,
        pos_label=1
    )

    fault_recall = recall_score(
        y_test,
        y_pred,
        pos_label=0
    )

    inference_time = measure_inference_time_tflite(
        model_path,
        X_timing,
        runs=100
    )

    size_info = get_parameter_size(model_path)

    return {
        "Accuracy": accuracy,
        "FDIA Recall": fdia_recall,
        "Fault Recall": fault_recall,
        "Inference Time (ms)": inference_time,
        "Parameters": size_info["params"],
        "FP32 Parameters": size_info["float32_params"],
        "INT8 Parameters": size_info["int8_params"],
        "Model Size (KB)": size_info["size_kb"]
    }

In [10]:
models = {
    "Original": {
        "Baseline": "LSTM_Original_baseline.tflite",
        "Quantized": "LSTM_Original_quant.tflite"
    },
    "Node": {
        "Baseline": "LSTM_Node_baseline.tflite",
        "Quantized": "LSTM_Node_quant.tflite"
    },
    "Weight": {
        "Baseline": "LSTM_Weight_baseline.tflite",
        "Quantized": "LSTM_Weight_quant.tflite"
    },
    "WeightNode": {
        "Baseline": "LSTM_WeightNode_baseline.tflite",
        "Quantized": "LSTM_WeightNode_quant.tflite"
    },
    "NodeWeight": {
        "Baseline": "LSTM_NodeWeight_baseline.tflite",
        "Quantized": "LSTM_NodeWeight_quant.tflite"
    }
}

results = []

for variant, versions in models.items():
    for version, filename in versions.items():
        print(f"\nTesting: {variant} - {version}")

        model_path = TFLITE_DIR / filename
        metrics = evaluate_tflite(model_path)

        results.append({
            "Model": variant,
            "Version": version,
            **metrics
        })

        print(f"Accuracy:        {metrics['Accuracy']:.6f}")
        print(f"FDIA Recall:     {metrics['FDIA Recall']:.6f}")
        print(f"Fault Recall:    {metrics['Fault Recall']:.6f}")
        print(f"Inference Time:  {metrics['Inference Time (ms)']:.6f} ms")
        print(f"Parameters:      {metrics['Parameters']:,}")
        print(f"FP32 Parameters: {metrics['FP32 Parameters']:,}")
        print(f"INT8 Parameters: {metrics['INT8 Parameters']:,}")
        print(f"Model Size:      {metrics['Model Size (KB)']:.3f} KB")

results_df = pd.DataFrame(results)

results_df


Testing: Original - Baseline


c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


KeyboardInterrupt: 

# MLP TESTS

In [24]:
X_timing_mlp = X_test[timing_idx].astype(np.float32)

In [27]:
def prepare_input(interpreter, X):
    input_details = interpreter.get_input_details()[0]
    dtype = input_details["dtype"]

    if dtype == np.float32:
        return X.astype(np.float32)

    scale, zero_point = input_details["quantization"]

    X_quant = X / scale + zero_point
    X_quant = np.round(X_quant)

    if dtype == np.uint8:
        X_quant = np.clip(X_quant, 0, 255)

    elif dtype == np.int8:
        X_quant = np.clip(X_quant, -128, 127)

    return X_quant.astype(dtype)


def predict_mlp_tflite(model_path, X):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))

    input_index = interpreter.get_input_details()[0]["index"]
    interpreter.resize_tensor_input(input_index, [len(X), 6, 83])
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    X_input = prepare_input(interpreter, X)

    interpreter.set_tensor(input_details["index"], X_input)
    interpreter.invoke()

    predictions = interpreter.get_tensor(output_details["index"])

    # Dequantize output if necessary
    if output_details["dtype"] != np.float32:
        scale, zero_point = output_details["quantization"]
        predictions = (predictions.astype(np.float32) - zero_point) * scale

    return predictions.squeeze()


def measure_mlp_tflite_time(model_path, X_timing, runs=100):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))

    input_index = interpreter.get_input_details()[0]["index"]
    interpreter.resize_tensor_input(input_index, [len(X_timing), 6, 83])
    interpreter.allocate_tensors()

    input_index = interpreter.get_input_details()[0]["index"]

    X_input = prepare_input(interpreter, X_timing)

    interpreter.set_tensor(input_index, X_input)
    interpreter.invoke()  # Warm-up

    run_times = []

    for _ in range(runs):
        interpreter.set_tensor(input_index, X_input)

        start = timeit.default_timer()
        interpreter.invoke()
        end = timeit.default_timer()

        run_times.append(((end - start) * 1000) / len(X_timing))

    return np.mean(run_times)

In [29]:
model_path = TFLITE_DIR / "MLP_Original_quant.tflite"

interpreter = tf.lite.Interpreter(model_path=str(model_path))

input_details = interpreter.get_input_details()[0]
input_index = input_details["index"]

interpreter.resize_tensor_input(
    input_index,
    [len(X_timing_mlp), 6, 83]
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
input_index = input_details["index"]

scale, zero_point = input_details["quantization"]

X_batch_uint8 = np.round(
    X_timing_mlp / scale + zero_point
)

X_batch_uint8 = np.clip(
    X_batch_uint8,
    0,
    255
).astype(np.uint8)

run_times = []

for _ in range(100):
    interpreter.set_tensor(input_index, X_batch_uint8)

    start = timeit.default_timer()
    interpreter.invoke()
    end = timeit.default_timer()

    run_times.append(
        ((end - start) * 1000) / len(X_timing_mlp)
    )

print(f"Mean inference time: {np.mean(run_times):.6f} ms")

Mean inference time: 0.007105 ms
